# 06 · item2vec và vector phiên (ticket #15)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/item2vec_{val,test}_run0.parquet`
- `item_embeddings/item2vec.npy`, `item_embeddings/item2vec_tracks.parquet`
- `item2vec_params.json`: tham số đã chọn và NDCG@10 trên val của cả lưới (lần chạy 1, 2 ở #26 dùng lại file này qua `--params`)

Word2Vec skip-gram (gensim) trên chuỗi bài của mỗi playlist train; negative 5, mọi bài train đều có vector. Vector phiên và điểm gu giống ALS (#14, `src/models/session.py`).

Lưới chọn trên val (vòng 2): window 3 × vector_size 128 × epochs {5, 20} × ns_exponent {0,75; −0,5} × normalize {không, có} × decay {0,5; 0,7; 0,9; 1,0} (32 cấu hình, 4 lần học Word2Vec; normalize và decay chỉ đổi cách chấm điểm). Chọn theo NDCG@10. Vòng 1 ở mục Kết quả.

`normalize`: chuẩn hóa mọi vector bài về độ dài 1 trước khi tính vector phiên và điểm gu, tức điểm gu là tổng có trọng số của cosine thay vì tích vô hướng thô (lệch nhỏ so với spec §6, ghi trong báo cáo). File `item2vec.npy` khi đó chứa vector đã chuẩn hóa, nên app vẫn chỉ cần tích vô hướng.

Học nhiều luồng nên cùng `random_seed` không ra cùng vector từng bit; khác biệt giữa các lần chạy đo ở #26.

gensim có sẵn trên Kaggle (#2). Bật **Internet** để clone repo. Chỉ cần CPU. Ước tính 35–45 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #15 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.item2vec import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # chọn tham số trên val, xuất file, có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lưới trên val, xếp theo NDCG@10, so với baseline (#9) và ALS (#14), cùng 9.212 phiên val
import pandas as pd

REFERENCE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297, "als": 0.0967}
grid = pd.DataFrame(report["grid"]).sort_values("ndcg", ascending=False)
print(grid.round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nChọn: {report['params']} | NDCG@10 val {ndcg:.4f}")
for name, ref in REFERENCE_NDCG.items():
    print(f"  so với {name} {ref}: {ndcg / ref - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Tab Output của version vừa chạy → **New Dataset** tên `mixnext-item2vec`; chỉ chọn thư mục `out`, bỏ `MusicRecSys`.
   - Không thêm được vào `mixnext-embeddings`: Dataset tạo từ output notebook chỉ nhận version mới từ chính notebook đó (notebook 05).
   - Notebook sau Add Input bộ này và tìm file bằng `Path("/kaggle/input").rglob(<tên file>)`.

## Kết quả trên val

### Vòng 1 (Kaggle, CPU, 26/11/2026, 1.754 giây): window {3, 10} × vector_size {128, 256} × decay, tham số mặc định của gensim (epochs 5, ns_exponent 0,75), tích vô hướng thô

| window | vector_size | decay | HitRate@10 | NDCG@10 | MRR |
|---:|---:|---:|---:|---:|---:|
| 3 | 128 | 0,5 | 0,0556 | **0,0282** | 0,0248 |
| 3 | 256 | 0,5 | 0,0526 | 0,0272 | 0,0244 |
| 10 | 256 | 0,5 | 0,0532 | 0,0266 | 0,0233 |
| 10 | 128 | 0,5 | 0,0521 | 0,0265 | 0,0237 |
| 3 | 128 | 0,7 | 0,0506 | 0,0256 | 0,0229 |
| 3 | 256 | 0,7 | 0,0481 | 0,0240 | 0,0216 |
| 10 | 128 | 0,7 | 0,0485 | 0,0236 | 0,0206 |
| 10 | 256 | 0,7 | 0,0479 | 0,0233 | 0,0205 |
| 3 | 128 | 0,9 | 0,0461 | 0,0228 | 0,0205 |
| 3 | 128 | 1,0 | 0,0448 | 0,0221 | 0,0199 |
| 3 | 256 | 0,9 | 0,0448 | 0,0219 | 0,0197 |
| 10 | 128 | 0,9 | 0,0433 | 0,0212 | 0,0189 |
| 3 | 256 | 1,0 | 0,0430 | 0,0207 | 0,0186 |
| 10 | 256 | 0,9 | 0,0422 | 0,0207 | 0,0186 |
| 10 | 128 | 1,0 | 0,0417 | 0,0206 | 0,0184 |
| 10 | 256 | 1,0 | 0,0404 | 0,0199 | 0,0179 |

Tốt nhất (3; 128; 0,5): NDCG@10 val 0,0282, hơn Popularity (0,0040) 7 lần nhưng kém ALS (0,0967) 71% và item-kNN liền kề (0,1297) 78%.

- window và vector_size gần như không ảnh hưởng (chênh khoảng 0,001), nên mở rộng hai tham số này không đáng; vòng 2 giữ (3, 128).
- decay 0,5 tốt nhất ở mọi cặp, giống ALS: vector không bị dồn về một hướng chung như trên dữ liệu giả.
- Nghi vấn: tích vô hướng thô bị độ dài vector (đi theo tần suất bài) chi phối, và tham số mặc định của gensim đặt cho văn bản. Vòng 2 thử chuẩn hóa vector, epochs 20 và ns_exponent âm (Caselles-Dupré và cộng sự 2018, "Word2vec applied to recommendation: hyperparameters matter").

### Vòng 2 (chưa chạy): epochs {5, 20} × ns_exponent {0,75; −0,5} × normalize {không, có} × decay

Dán bảng lưới và dòng "Chọn: …" vào đây sau khi chạy. Lưới có cấu hình tốt nhất của vòng 1 (epochs 5, ns_exponent 0,75, không chuẩn hóa, decay 0,5), nên kết quả vòng 2 không kém vòng 1 (trừ sai khác do học nhiều luồng).

Giới hạn: `item2vec.npy` phải đủ nhỏ để `artifacts/` của app dưới 200 MB (#20); 128 chiều khoảng 67 MB. decay giữ đúng tập của spec §6.